In [1]:
import duckdb
p = r"C:\Users\eddiec11us\dev_apps\reporter\databases\status_reports.duckdb"
conn = duckdb.connect(p)

In [ ]:
schema = ('acct_num', 'part_number', 'qty', 'amount', 'invoice_date', 'part_category', 'year')

In [ ]:
"""
Month over Month; even if a value is zero
"""

import pandas as pd
dates_tbl = pd.DataFrame(columns=["dates"], data=pd.date_range(start="1/1/2024", end="12/31/2025", freq='ME'))
dates_tbl = dates_tbl.assign(month=dates_tbl["dates"].dt.month, year=dates_tbl["dates"].dt.year)
duckdb.sql("CREATE OR REPLACE TABLE dates_tbl AS SELECT * FROM dates_tbl")
duckdb.sql("INSERT INTO dates_tbl SELECT * FROM dates_tbl")

accts_tbl = pd.DataFrame(columns=['acct_num'], data=['STA810401'])
duckdb.sql("CREATE OR REPLACE TABLE accts_tbl AS SELECT * FROM accts_tbl")
duckdb.sql("INSERT INTO accts_tbl SELECT * FROM accts_tbl")

In [ ]:
schema = ('acct_num', 'part_number', 'qty', 'amount', 'invoice_date', 'part_category', 'year')


# Monthly sales (long data) 
 
q = """
WITH acct_base AS (
  SELECT 
    accts.acct_num,
    dates.month,
    dates.year  
  FROM accts_tbl accts
  CROSS JOIN dates_tbl AS dates
), 
grp_sales AS (
  SELECT
    s.acct_num,
    ROUND(SUM(s.amount)) AS month_net,
    EXTRACT (month FROM invoice_date) AS invoice_month, 
    EXTRACT (year FROM invoice_date) AS invoice_year
  FROM sales AS s
  GROUP BY 
    s.acct_num, 
    invoice_month, 
    invoice_year
)
SELECT 
  ab.acct_num,
  ab.month,
  ab.year,
  gs.month_net,
  gs.invoice_month,
  gs.invoice_year
FROM acct_base ab
LEFT JOIN grp_sales AS gs 
  ON  ab.acct_num = gs.acct_num
  AND ab.month    = gs.invoice_month
  AND ab.year     = gs.invoice_year 
ORDER BY 
  ab.year, 
  ab.month;
"""

df = conn.query(query=q).df()

df


,acct_num,month,year,month_net,invoice_month,invoice_year
0,STA810401,1,2024,19020.0,1,2024
1,STA810401,2,2024,33283.0,2,2024
2,STA810401,3,2024,11227.0,3,2024
3,STA810401,4,2024,28317.0,4,2024
4,STA810401,5,2024,8991.0,5,2024
5,STA810401,6,2024,18849.0,6,2024
6,STA810401,7,2024,14175.0,7,2024
7,STA810401,8,2024,23066.0,8,2024
8,STA810401,9,2024,9935.0,9,2024
9,STA810401,10,2024,27952.0,10,2024
